# 질감 게이트 v2 + 베드 마스킹으로 다시 학습 — ConvNeXt-Tiny, 시드 0·1

**무엇이 바뀌나:** 지난번 실험(고정 크롭 + 기존 Otsu 게이트 0.25)과 모두 같고 두 가지만 바꾼다.
1. **게이트**: 팀원님 질감 게이트 v2(0.1). 걸린 사진(약 551장)은 학습에서도 평가에서도 빠진다.
2. **마스킹**: 게이트를 통과한 사진에서 출력물 바깥(베드)을 회색으로 칠한 뒤 모델에 넣는다. 베드 반짝이 때문에 정상을 언더필로 잘못 잡는 오탐을 줄이려는 것이다.

실제로 쓸 때와 같은 조건이다(판정할 때도 같은 게이트와 마스킹을 거친다).

- 모델: ConvNeXt-Tiny, finetune 20 epoch, 시간 블록 5-fold
- 시드 0 → 1 순서. 시드 하나에 약 1시간 25분. 시드마다 결과가 따로 저장되니 시간이 모자라면 시드 0까지만 써도 된다.

**Drive `산공종설/`에 필요한 파일**
- `underfill_originals_4000.zip`: 이미 있음
- `pipeline_train_bundle_20261002_texture.zip`: **새로 올릴 것**. 학습 스크립트(`--gate`, `--mask-bed` 옵션 추가본), 팀원님 `filament_gate.py` v2(수정 없음), 분할표

노트북은 **파일 > 노트북 업로드**로 연다. 1번부터 순서대로 실행한다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS = f"{DRIVE_DIR}/results/pipeline_fc512_texture_v2_mask"
OLD = f"{DRIVE_DIR}/results/pipeline_fc512_gate"          # 지난번(기존 게이트) 결과, 비교용
os.makedirs(RESULTS, exist_ok=True)
for f in ("underfill_originals_4000.zip", "pipeline_train_bundle_20261002_mask_v2.zip"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 압축 해제
`v2-mask`가 보이는 줄과, 마지막 줄에 4000이 나와야 한다.

In [ ]:
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/pipeline_train_bundle_20261002_mask_v2.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/pipeline_train_bundle_20261002_mask_v2.zip -d /content/code
!grep -m1 VERSION /content/code/filament_gate.py
!ls /content/code
!find /content/orig -name "*.jpg" | wc -l

## 4. 본 실행 (시드 0 → 1)
- `--gate texture --min-filament 0.1`: 질감 게이트로 출력물 면적이 10% 미만인 사진을 뺀다.
- `--mask-bed`: 남은 사진의 베드를 회색으로 칠한다. 나머지 옵션은 지난번과 같다.
- 시작하고 몇 분 뒤 `gate texture (filament < 0.1): held 551 images {'normal': 257, 'underfill_50': 294}` 줄이 나오면 v2 게이트가 제대로 붙은 것이다.
- fold 하나가 끝날 때마다 한 줄씩 나온다(약 15분 간격).
- **끊겼을 때:** 이 셀을 다시 실행하면 끝난 시드는 건너뛰고 남은 것부터 한다.
- **시간이 모자라면:** 시드 1 도중에 셀을 멈추고(■) 5번부터 실행한다. 시드 0 결과만으로 정리된다.

In [ ]:
import shutil
COMMON = ("--folds-csv /content/code/folds.csv --pilot-root /content/orig --input-column source "
          "--fixed-crop 0 250 1000 1250 --img-size 512 --modes finetune --fast --gate texture --min-filament 0.1 --mask-bed")
HIDE = r"^ \|^{\|^}"     # 학습 끝의 긴 요약(JSON)을 화면에서 숨김. 같은 내용이 CSV에 있음
ARCH, SEEDS = "convnext_tiny", [0, 1]
for SEED in SEEDS:
    OUT = f"{RESULTS}/{ARCH}_seed{SEED}"
    if os.path.exists(f"{OUT}/summary.csv"):
        print("이미 끝남:", OUT)
        continue
    shutil.rmtree(OUT, ignore_errors=True)   # 반만 끝난 폴더 정리
    print(f"\n===== {ARCH} seed {SEED}")
    !PYTHONUNBUFFERED=1 python /content/code/train_pilot_classifier.py {COMMON} --arch {ARCH} --seeds {SEED} --output "{OUT}" | grep -v '{HIDE}'

## 5. 결과: 기존 게이트 vs 질감 게이트 v2 + 마스킹 (같은 시드끼리)
- `기존 게이트`: 지난번 결과(3,951장)
- `v2+마스킹`: 이번 결과(약 3,449장, 판정 비율 약 86%)
- 판정한 사진 수가 다르니 `n`(평가 사진 수)을 같이 본다. `vote_…`는 최근 5장 다수결 점수다.

**이 셀 출력을 복사해서 Claude에게 붙여 주면 된다.**

In [ ]:
import pandas as pd
rows = []
for SEED in SEEDS:
    for name, base in (("기존 게이트", OLD), ("v2+마스킹", RESULTS)):
        OUT = f"{base}/{ARCH}_seed{SEED}"
        if not os.path.exists(f"{OUT}/summary.csv"):
            continue
        s = pd.read_csv(f"{OUT}/summary.csv", encoding="utf-8-sig").iloc[0]
        v = pd.read_csv(f"{OUT}/vote_metrics.csv", encoding="utf-8-sig").iloc[0]
        rows.append({"seed": SEED, "gate": name, "n": int(s[["pooled_tn_sum_over_seeds", "pooled_fp_sum_over_seeds", "pooled_fn_sum_over_seeds", "pooled_tp_sum_over_seeds"]].sum()),
                     "acc": s.pooled_mean_accuracy, "auroc": s.pooled_mean_auroc,
                     "underfill_recall": s.pooled_mean_underfill_recall, "normal_recall": s.pooled_mean_normal_recall,
                     "FP": int(s.pooled_fp_sum_over_seeds), "FN": int(s.pooled_fn_sum_over_seeds),
                     "fold_std_acc": s.fold_std_accuracy, "vote_acc": v.vote_accuracy})
print(pd.DataFrame(rows).round(3).to_string(index=False) if rows else "아직 끝난 시드가 없음")
for SEED in SEEDS:
    f = f"{RESULTS}/{ARCH}_seed{SEED}/fold_metrics.csv"
    if os.path.exists(f):
        fm = pd.read_csv(f, encoding="utf-8-sig")
        print(f"\n[v2+마스킹 seed {SEED}] fold별 정상 재현율", fm.val_normal_recall.round(3).tolist(),
              "/ 언더필 재현율", fm.val_underfill_recall.round(3).tolist())

## 6. 결과 CSV 묶기
가중치를 뺀 CSV만 Drive `results/pipeline_fc512_texture_v2_mask_csv.zip`으로 묶는다. 이 zip을 받아서 Claude에게 주면 된다.

In [ ]:
import zipfile, glob
ZIP = f"{DRIVE_DIR}/results/pipeline_fc512_texture_v2_mask_csv.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{RESULTS}/**/*", recursive=True):
        if p.endswith((".csv", ".json", ".txt")):
            z.write(p, os.path.relpath(p, RESULTS))
print(ZIP, f"{os.path.getsize(ZIP) / 2**20:.1f} MiB")

## 7. 런타임 반납

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()